# 06. RFM-сегментация

В предыдущих ноутбуках клиенты разделялись по одному признаку (размеру первого чека), что обеспечивало интерпретируемость анализа, но ограничивало детальность сегментации. RFM расширяет подход за счёт одновременного учёта трёх поведенческих осей:

- R (Recency): количество дней с момента последней покупки. Меньшее значение интерпретируется как более активное состояние клиента.
- F (Frequency): общее число заказов клиента за период наблюдения.
- M (Monetary): совокупная выручка с клиента.

По каждой оси клиенты разбиваются на квинтили (1 = наименее благоприятный класс, 5 = наиболее благоприятный). Комбинация трёх оценок задаёт RFM-код (например, 555 для топ-сегмента или 111 для худшего). Поверх кодов накладывается интерпретируемая бизнес-разметка: «Чемпионы», «Лояльные», «Под угрозой», «Потерянные» и так далее.

Ключевое преимущество подхода: каждому сегменту соответствует операционно осмысленное действие, что делает результат непосредственно применимым в работе CRM-команды.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet('../data/clean.parquet')
print(f'Транзакций: {len(df):,}')

## Расчёт показателей R, F, M

В качестве точки отсчёта для recency используется дата последней транзакции в датасете плюс один день. Такой подход исключает нулевые значения у самых свежих клиентов и является стандартным для офлайн-расчётов на исторических данных.

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Snapshot date: {snapshot_date:%Y-%m-%d}')

rfm = (
    df.groupby('Customer ID')
    .agg(
        recency=('InvoiceDate', lambda s: (snapshot_date - s.max()).days),
        frequency=('Invoice', 'nunique'),
        monetary=('Revenue', 'sum'),
    )
    .reset_index()
)
rfm.head()

## Назначение баллов от 1 до 5

Шкала для recency инвертируется: низкие значения recency соответствуют высокой оценке (5), поскольку отражают недавнюю активность клиента. В частотном и денежном измерениях прямая шкала: большие значения соответствуют большим оценкам.

In [ ]:
# pd.qcut по умолчанию делит на квинтили по содержимому. У frequency распределение содержит
# большое число повторяющихся значений (значительная часть клиентов имеет 1-2 заказа), поэтому
# напрямую квантильное разбиение приводит к ошибке о неуникальных границах.
# Применение rank(method='first') устраняет проблему, разнося ранги по уникальным позициям.
rfm['R_score'] = pd.qcut(rfm['recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm.head()

## Бизнес-разметка сегментов

Существует несколько устоявшихся схем разметки RFM. Используется компактная и интерпретируемая версия, основанная на следующих правилах:
- Высокие R и F: клиент находится в активной фазе.
- Низкий R при ранее высоком F: бывший лояльный клиент, кандидат на реактивацию.
- Низкие R и F одновременно: «потерянный» сегмент с минимальной экономической отдачей от воздействия.

In [ ]:
def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4:
        return 'Чемпионы'
    if r >= 3 and f >= 3:
        return 'Лояльные'
    if r >= 4 and f <= 2:
        return 'Новички'
    if r == 3 and f <= 2:
        return 'Перспективные'
    if r <= 2 and f >= 4:
        return 'Под угрозой'
    if r <= 2 and f >= 2:
        return 'Спящие'
    return 'Потерянные'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

segment_summary = (
    rfm.groupby('segment')
    .agg(
        n_customers=('Customer ID', 'count'),
        avg_recency_days=('recency', 'mean'),
        avg_frequency=('frequency', 'mean'),
        avg_monetary=('monetary', 'mean'),
        total_monetary=('monetary', 'sum'),
    )
    .round(1)
    .sort_values('total_monetary', ascending=False)
)
segment_summary['share_revenue_pct'] = (segment_summary['total_monetary'] / segment_summary['total_monetary'].sum() * 100).round(1)
segment_summary

Качественные наблюдения по итоговой таблице:
1. Сегмент «Чемпионы» относительно невелик в количественном выражении, но обеспечивает непропорционально высокую долю выручки. Это та самая «голова» распределения, выявленная в ноутбуке 04 при анализе концентрации.
2. Сегмент «Под угрозой» представляет наибольший практический интерес для retention-кампаний: клиенты ранее демонстрировали высокую активность, но давно не совершали покупок. Бренд для них ещё актуален, и точечное воздействие имеет высокую вероятность отклика.
3. Сегмент «Потерянные» характеризуется низким ожидаемым ROI коммуникаций. Маркетинговый бюджет на этот сегмент следует ограничивать.

In [ ]:
# Сравнение доли клиентов и доли выручки по сегментам.
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(segment_summary))
width = 0.4
share_customers = segment_summary['n_customers'] / segment_summary['n_customers'].sum() * 100
share_revenue = segment_summary['share_revenue_pct']

ax.bar(x - width/2, share_customers, width, label='доля клиентов, %', color='#4C72B0')
ax.bar(x + width/2, share_revenue, width, label='доля выручки, %', color='#55A868')
ax.set_xticks(x)
ax.set_xticklabels(segment_summary.index, rotation=20)
ax.set_ylabel('%')
ax.set_title('RFM-сегменты: доля клиентов и доля выручки')
ax.legend()
plt.tight_layout()
plt.savefig('../images/rfm_segments.png', dpi=120, bbox_inches='tight')
plt.show()

Интерпретация графика. Сегменты, в которых доля выручки превышает долю клиентов, обладают повышенной экономической ценностью на единицу аудитории; на них имеет смысл концентрировать ресурсы. Сегменты с обратным соотношением представляют большую часть базы при низкой удельной отдаче и требуют осторожного подхода к стимулированию.

## План работы по сегментам

Сегментация служит основой для CRM-плана с привязкой к конкретным действиям:

- Чемпионы: программы лояльности, ранний доступ к новинкам, отказ от массовых скидок (текущая активность не требует стимулирования).
- Лояльные: апсейл по сопутствующим категориям, развитие через подписочные продукты или программы членства.
- Под угрозой: персонализированные реактивационные коммуникации. На этом сегменте триггерные кампании дают наибольший ожидаемый ROI.
- Спящие: широкая реактивационная кампания с увеличенным размером скидки, без точного таргетинга по интересам (актуальная информация о предпочтениях недостаточна).
- Новички: онбординг, фокус на UX и стимулировании второй покупки. Рекомендация ноутбука 03 (триггерная кампания на низкочековых клиентов) применяется именно к этому сегменту.
- Перспективные: стимулирование третьей и четвёртой покупки с целью перевода в категорию «Лояльные».
- Потерянные: воздействие минимизируется. Допустим однократный контентный win-back без скидок для проверки реакции.

## Сохранение результатов

In [ ]:
rfm.to_parquet('../data/rfm.parquet', index=False)
print('Сохранено: data/rfm.parquet')

## Резюме

RFM представляет собой инструмент сегментирования базы по поведенческим характеристикам, не относящийся ни к статистическому моделированию, ни к машинному обучению. Сильная сторона метода: каждый сегмент допускает прямой перевод в маркетинговое действие. Ограничение, требующее учёта в продакшен-эксплуатации: границы квинтилей зависят от текущего распределения данных, поэтому при изменении структуры трафика шкалы необходимо пересчитывать. На практике RFM-таблица обновляется с регулярной периодичностью (еженедельно или ежемесячно).